In [1]:
# search_hnsw.py
import hnswlib
import pickle
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
from feature_extraction import extract_features # Your existing function

# --- Configuration ---
index_file = 'sweater_hnsw_ResNet50_Individual.bin'
pattern_ids_file = 'all_image_pattern_ids.pkl'
feature_dim = 1024 # Should match the dimension used for building the index

def recommend_similar_images(query_image_path, hnsw_index, pattern_ids, num_recommendations=5):
    """
    Finds and displays the most similar images to a query image using an HNSWlib index.
    """
    print(f"--- Generating recommendations for {os.path.basename(query_image_path)} ---")
    
    # --- 1. Extract features from the query image ---
    _, query_vector = extract_features(query_image_path)
    if query_vector is None or not isinstance(query_vector, np.ndarray):
        print(f"Could not extract features from {query_image_path}")
        return

    # HNSWlib requires a 1D or 2D array, and it must be float32
    query_vector = np.array(query_vector).astype('float32')
    
    # --- 2. Search the HNSWlib index ---
    # The knn_query method returns labels and distances for the k nearest neighbors
    labels, distances = hnsw_index.knn_query(query_vector, k=num_recommendations)

    # The result is a 2D array, so we get the first (and only) row
    neighbor_indices = labels[0]
    
    print(f"Found neighbors with indices: {neighbor_indices}")
    
    # --- 3. Display the results ---
    plt.figure(figsize=(15, 5))

    # Display Query Image
    ax = plt.subplot(1, num_recommendations + 1, 1)
    query_img = Image.open(query_image_path)
    ax.imshow(query_img)
    ax.set_title("Query Image")
    ax.axis("off")

    # Display Recommended Images
    for i, idx in enumerate(neighbor_indices):
        recommended_pattern_id = pattern_ids[idx]
        
        # IMPORTANT: Update this path to how your images are stored.
        # This example assumes a structure like: data_directory/pattern_id/image.jpg
        rec_img_folder = os.path.join('/Volumes/Extreme Pro/ANN_photos', recommended_pattern_id)
        
        # Find the first image in the recommended pattern folder to display
        try:
            image_files = [f for f in os.listdir(rec_img_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if not image_files:
                print(f"Warning: No images found for pattern {recommended_pattern_id}.")
                continue
            first_image_file = image_files[0]
            rec_img_path = os.path.join(rec_img_folder, first_image_file)

            ax = plt.subplot(1, num_recommendations + 1, i + 2)
            rec_img = Image.open(rec_img_path)
            ax.imshow(rec_img)
            ax.set_title(f'Rec #{i+1}\n{recommended_pattern_id}')
            ax.axis('off')
        except (IOError, IndexError) as e:
            print(f"Warning: Could not load image for pattern {recommended_pattern_id}. Error: {e}")


    plt.tight_layout()
    plt.show()



Initializing YOLOv8 models...
YOLOv8n model initialized.


In [5]:
import os
import pickle
import hnswlib
import numpy as np
import time

# --- Configuration ---
test_image_folder = 'Evaluation_Sweaters/pullovers'
valid_image_extensions = ('.jpg', '.jpeg', '.png')

# --- Configuration for Unique Recall ---
target_unique_count = 10   # We want the top 10 UNIQUE patterns
k_fetch = 100              # Fetch more raw neighbors to ensure we find 10 unique ones

# --- Lists to store results ---
all_query_times_ms = []
all_hits = [] 

if not os.path.isdir(test_image_folder):
    print(f"Error: Test image folder not found at '{test_image_folder}'")
else:
    print(f"\n--- Starting batch recommendation for all images in {test_image_folder} ---")
    
    image_files_to_process = [
        f for f in sorted(os.listdir(test_image_folder)) 
        if f.lower().endswith(valid_image_extensions)
    ]
    total_images = len(image_files_to_process)
    
    for i, image_filename in enumerate(image_files_to_process):
        test_image_path = os.path.join(test_image_folder, image_filename)
        
        try:
            # --- Step 1: Get Ground Truth ID ---
            id_from_file = image_filename.split('.')[0]
            correct_pattern_id = f"pattern_{id_from_file}"
            
            # --- Step 2: Extract features ---
            img_path_out, query_vector = extract_features(test_image_path)
            
            if not isinstance(query_vector, np.ndarray):
                continue 

            # --- Step 3: Run Query (Fetch larger batch) ---
            start_time = time.perf_counter()
            # We fetch 'k_fetch' (e.g. 100) instead of just 10
            labels, distances = hnsw_index.knn_query(query_vector, k=k_fetch)
            end_time = time.perf_counter()
            
            query_time_ms = (end_time - start_time) * 1000 
            all_query_times_ms.append(query_time_ms)

            # --- Step 4: Deduplicate to get Top 10 Unique ---
            unique_recommendations = []
            seen_patterns = set()
            
            # Iterate through the raw neighbors
            for label_idx in labels[0]:
                pat_id = pattern_ids[label_idx]
                
                if pat_id not in seen_patterns:
                    unique_recommendations.append(pat_id)
                    seen_patterns.add(pat_id)
                
                # Stop once we have our target number (10)
                if len(unique_recommendations) >= target_unique_count:
                    break
            
            # --- Step 5: Check for Hit ---
            print(f"Testing {image_filename} -> GT: {correct_pattern_id}")
            # print(f"  > Unique Recs: {unique_recommendations}") # Uncomment to see full list
            
            if correct_pattern_id in unique_recommendations:
                all_hits.append(1)
                print(f"  > RESULT: HIT! Found in top {len(unique_recommendations)} unique.")
            else:
                all_hits.append(0)
                print(f"  > RESULT: MISS.")
                
        except Exception as e:
            print(f"ERROR processing {image_filename}: {e}")

    # --- Final Stats ---
    if all_hits:
        total_processed = len(all_hits)
        num_hits = np.sum(all_hits)
        avg_accuracy = (num_hits / total_processed) * 100
        print(f"\n=== RESULTS (Recall @ {target_unique_count} Unique Patterns) ===")
        print(f"Total Images: {total_processed}")
        print(f"Total Hits:   {num_hits}")
        print(f"Accuracy:     {avg_accuracy:.2f}%")


--- Starting batch recommendation for all images in Evaluation_Sweaters/pullovers ---
  -> Final image shape: (336, 379, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Testing 1001132.jpeg -> GT: pattern_1001132
  > RESULT: MISS.
  -> Final image shape: (434, 463, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Testing 100437.jpeg -> GT: pattern_100437
  > RESULT: HIT! Found in top 10 unique.
  -> Final image shape: (260, 315, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Testing 1021796.jpeg -> GT: pattern_1021796
  > RESULT: MISS.
  -> No masks or boxes found. Falling back to original image.
  -> Final image shape: (640, 457, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Testing 1037791.jpeg -> GT: pattern_1037791
  > RESULT: MISS.
  -> Final image shape: (441, 520, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Testing 1066129.jpeg -> GT: pattern_1066129
  > RESULT: MISS.
  -> Final image shape: (263, 265, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Testing 1094188.png -> GT: pattern_1094188
  > RESULT: MISS.
  -

In [6]:
import os
import pickle
import hnswlib
import numpy as np
import time

# --- Configuration ---
test_image_folder = 'Evaluation_Sweaters/cardigans'
valid_image_extensions = ('.jpg', '.jpeg', '.png')

num_recommendations = 30


# ... (Imports and Setup remain the same) ...

# --- Configuration for Unique Recall ---
target_unique_count = 10   # We want the top 10 UNIQUE patterns
k_fetch = 100              # Fetch more raw neighbors to ensure we find 10 unique ones

# --- Lists to store results ---
all_query_times_ms = []
all_hits = [] 

if not os.path.isdir(test_image_folder):
    print(f"Error: Test image folder not found at '{test_image_folder}'")
else:
    print(f"\n--- Starting batch recommendation for all images in {test_image_folder} ---")
    
    image_files_to_process = [
        f for f in sorted(os.listdir(test_image_folder)) 
        if f.lower().endswith(valid_image_extensions)
    ]
    total_images = len(image_files_to_process)
    
    for i, image_filename in enumerate(image_files_to_process):
        test_image_path = os.path.join(test_image_folder, image_filename)
        
        try:
            # --- Step 1: Get Ground Truth ID ---
            id_from_file = image_filename.split('.')[0]
            correct_pattern_id = f"pattern_{id_from_file}"
            
            # --- Step 2: Extract features ---
            img_path_out, query_vector = extract_features(test_image_path)
            
            if not isinstance(query_vector, np.ndarray):
                continue 

            # --- Step 3: Run Query (Fetch larger batch) ---
            start_time = time.perf_counter()
            # We fetch 'k_fetch' (e.g. 100) instead of just 10
            labels, distances = hnsw_index.knn_query(query_vector, k=k_fetch)
            end_time = time.perf_counter()
            
            query_time_ms = (end_time - start_time) * 1000 
            all_query_times_ms.append(query_time_ms)

            # --- Step 4: Deduplicate to get Top 10 Unique ---
            unique_recommendations = []
            seen_patterns = set()
            
            # Iterate through the raw neighbors
            for label_idx in labels[0]:
                pat_id = pattern_ids[label_idx]
                
                if pat_id not in seen_patterns:
                    unique_recommendations.append(pat_id)
                    seen_patterns.add(pat_id)
                
                # Stop once we have our target number (10)
                if len(unique_recommendations) >= target_unique_count:
                    break
            
            # --- Step 5: Check for Hit ---
            print(f"Testing {image_filename} -> GT: {correct_pattern_id}")
            # print(f"  > Unique Recs: {unique_recommendations}") # Uncomment to see full list
            
            if correct_pattern_id in unique_recommendations:
                all_hits.append(1)
                print(f"  > RESULT: HIT! Found in top {len(unique_recommendations)} unique.")
            else:
                all_hits.append(0)
                print(f"  > RESULT: MISS.")
                
        except Exception as e:
            print(f"ERROR processing {image_filename}: {e}")

    # --- Final Stats ---
    if all_hits:
        total_processed = len(all_hits)
        num_hits = np.sum(all_hits)
        avg_accuracy = (num_hits / total_processed) * 100
        print(f"\n=== RESULTS (Recall @ {target_unique_count} Unique Patterns) ===")
        print(f"Total Images: {total_processed}")
        print(f"Total Hits:   {num_hits}")
        print(f"Accuracy:     {avg_accuracy:.2f}%")


--- Starting batch recommendation for all images in Evaluation_Sweaters/cardigans ---
  -> Final image shape: (420, 412, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Testing 100136.jpeg -> GT: pattern_100136
  > RESULT: MISS.
  -> No masks or boxes found. Falling back to original image.
  -> Final image shape: (480, 640, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Testing 1028254.jpeg -> GT: pattern_1028254
  > RESULT: MISS.
  -> Final image shape: (350, 438, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Testing 1030902.jpeg -> GT: pattern_1030902
  > RESULT: MISS.
  -> Final image shape: (423, 187, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Testing 103910.jpeg -> GT: pattern_103910
  > RESULT: MISS.
  -> Final image shape: (446, 514, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Testing 110719.jpg -> GT: pattern_110719
  > RESULT: MISS.
  -> Final image shape: (441, 433, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Testing 1119779.jpeg -> GT: pattern_1119779
  > RESULT: MISS.
  -> Final image shape: (368, 